# Reproduce the PPI benchmark audit
Run from the repository root with the PPI-stanza kernel.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys

ROOT = Path.cwd()
if not (ROOT / "src/reproduction/experiment.json").is_file():
    raise RuntimeError("Open reproduce.ipynb from the repository root.")
sys.path.insert(0, str(ROOT))
from src.reproduction.config import SOURCE_PAPER_DIR, CORPORA, stats_dir

OUTPUT = ROOT / "output" / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
print("Python:", sys.executable)
print("Paper:", SOURCE_PAPER_DIR)
print("New results:", OUTPUT)

## 2. Check saved results

In [ ]:
from src.tools.smoke import check_saved_results

check_saved_results()

## 3. Recompute tables

In [ ]:
import pandas as pd
from IPython.display import display
from src.analysis.statistical_tests import write_statistical_outputs
from src.reproduction.within_corpus import analyze

ledgers = [stats_dir(corpus) for corpus in CORPORA]
write_statistical_outputs(ledgers, OUTPUT / "cross_corpus", min_support=5)
cells, summary, examples = analyze(ledgers, min_support=5, isolated_frac=0.25)
pd.DataFrame(cells).to_csv(OUTPUT / "within_corpus_cells.csv", index=False)
pd.DataFrame(summary).to_csv(OUTPUT / "within_corpus_summary.csv", index=False)

comparisons = [
    ("cross_corpus/cpr_summary.csv", "statistical_tests/apbc/cpr_summary.csv"),
    ("cross_corpus/cross_corpus_fisher_bh.csv", "statistical_tests/apbc/cross_corpus_fisher_bh.csv"),
    ("within_corpus_summary.csv", "within_corpus/apbc/within_corpus_summary.csv"),
]
for generated, saved in comparisons:
    actual = pd.read_csv(OUTPUT / generated)
    expected = pd.read_csv(SOURCE_PAPER_DIR / "analysis" / saved)
    pd.testing.assert_frame_equal(actual, expected, check_dtype=False, rtol=1e-9, atol=1e-12)
    print(f"PASS: {generated}")
display(pd.DataFrame(summary))

## 4. Human ratings
One annotator, 163 items; rates exclude `uncertain`. Schema: [validation/README.txt](supplements/validation/README.txt).

In [ ]:
from src.reproduction.validate_human_ratings import load_public, QUESTIONS

ratings = pd.DataFrame(load_public())
rows = []
for question in QUESTIONS:
    counts = ratings[question].value_counts()
    yes, no = int(counts.get("yes", 0)), int(counts.get("no", 0))
    rows.append({"question": question, "yes": yes, "no": no,
                 "uncertain": int(counts.get("uncertain", 0)),
                 "definite": yes + no, "yes_fraction": yes / (yes + no)})
assert len(ratings) == 163
print("Independent ratings:", len(ratings))
display(pd.DataFrame(rows))

## 5. Parser check (optional)
Needs the parser models.

In [ ]:
RUN_PARSER_CHECK = False

if RUN_PARSER_CHECK:
    from src.tools.smoke import check_parser
    check_parser()
else:
    print("Skipped parser check.")

## 6. Rebuild (optional)
Set `REBUILD` to `"statistics"` or `"full"`; `full` needs the parser models.

In [ ]:
REBUILD = "none"  # "none", "statistics", or "full"

if REBUILD == "none":
    print("Skipped rebuild.")
elif REBUILD in {"statistics", "full"}:
    import json
    from src.reproduction.runner import reproduce
    report = reproduce(REBUILD, OUTPUT / REBUILD)
    assert json.loads(report.read_text())["status"] == "passed"
    print("Report and logs:", report.parent)
else:
    raise ValueError('Choose "none", "statistics", or "full".')